# REHAB processed dataset: all movements

This notebook joins the 32 arrays in `d02_processed_data` into one DataFrame. `movement_type` is the objective variable and `record_id` identifies each independent recording. It then generates `stroke_report.html` with `data_profiling`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("REHAB/Rehab_exercise/d02_processed_data").resolve()

MOVEMENTS = [
    "bobath_handshake",
    "bobath_flexion_extension",
    "bobath_forward_flexion_extension",
    "bobath_anterior_posterior_rotation",
    "elbow_flexion_wrist_compression",
    "wrist_flexion_extension",
    "finger_to_finger_training",
    "ball_gripping",
    "shoulder_internal_external_rotation",
    "breast_expansion",
    "flexion_pressure_rotation",
    "elbow_flexion_touch",
    "shoulder_touch_training",
    "ankle_extension_knee_rotation",
    "knee_flexion_extension",
    "hip_flexion_extension",
]

SENSOR_COLUMNS = [
    "pitch_1", "yaw_1", "roll_1",
    "pitch_2", "yaw_2", "roll_2",
    "finger_1", "finger_2", "finger_3",
    "finger_4", "finger_5", "wrist_pitch",
]

frames = []
next_record_id = 0

for movement_id, movement_name in enumerate(MOVEMENTS):
    imu = np.load(DATA_DIR / f"{movement_id:03d}_1.npy")
    glove = np.load(DATA_DIR / f"{movement_id:03d}_2.npy")

    if imu.shape != glove.shape or imu.shape[1:] != (880, 6):
        raise ValueError(f"Unexpected array shapes for movement {movement_id:03d}")

    record_count, timepoint_count, _ = imu.shape
    values = np.concatenate([imu, glove], axis=2).reshape(-1, 12)
    movement_frame = pd.DataFrame(values, columns=SENSOR_COLUMNS)
    movement_frame.insert(0, "time_s", np.tile(np.arange(timepoint_count) / 50, record_count))
    movement_frame.insert(0, "timepoint", np.tile(np.arange(timepoint_count), record_count))
    movement_frame.insert(0, "sample_id", np.repeat(np.arange(record_count), timepoint_count))
    movement_frame.insert(0, "record_id", np.repeat(np.arange(next_record_id, next_record_id + record_count), timepoint_count))
    movement_frame.insert(0, "movement_type", movement_name)
    frames.append(movement_frame)
    next_record_id += record_count

df = pd.concat(frames, ignore_index=True)
df["movement_type"] = pd.Categorical(df["movement_type"], categories=MOVEMENTS)

print(f"Processed data: {DATA_DIR}")
print(f"Dataset shape: {df.shape}")
print(f"Recordings: {df['record_id'].nunique()}")
print("Objective variable: movement_type")
df.head()

In [ ]:
from data_profiling import ProfileReport

profile = ProfileReport(
    df,
    title="REHAB Exercise Processed Dataset — All Movements",
    minimal=True,
    progress_bar=False,
    dataset={
        "description": (
            f"All {df['record_id'].nunique():,} processed REHAB recordings. "
            "The objective variable is movement_type and record_id identifies each recording."
        ),
        "url": "https://doi.org/10.1038/s41597-026-07802-2",
    },
)

REPORT_PATH = Path("stroke_report.html").resolve()
profile.to_file(REPORT_PATH)
print(f"Report written to: {REPORT_PATH}")